# RQ3: Amazon Toys Encoder Comparison

This notebook builds text embeddings from local Amazon Toys raw files, then runs fixed dVAE and a lightweight seqrec setup.

The raw `.jsonl.gz` files are read into RAM and decompressed in RAM before any parquet output is written. This avoids slow repeated disk reads on the Jupyter machine.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/home/jupyter/project/semantic_ids')
if not ROOT.exists():
    ROOT = Path.cwd().resolve()
os.chdir(ROOT)

PY = sys.executable
REVIEWS_GZ = ROOT / 'data' / 'Toys_and_Games.jsonl.gz'
META_GZ = ROOT / 'data' / 'meta_Toys_and_Games.jsonl.gz'
RAW_OUT = ROOT / 'data' / 'amazon_toys_encoders'
RESULTS_DIR = ROOT / 'results' / 'RQ_encoder_comparison_amazon_toys'

print('ROOT:', ROOT)
print('REVIEWS_GZ:', REVIEWS_GZ, REVIEWS_GZ.exists())
print('META_GZ:', META_GZ, META_GZ.exists())
print('HF token present:', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))

## 1. Build Interactions And Embeddings

Default encoders:

- `tiny_minilm_l3`: small/simple baseline
- `minilm_l6`: common stronger MiniLM baseline
- `bge_small`: stronger small BGE encoder

Embeddings are computed with pure `transformers` + `torch` mean pooling, not `sentence_transformers`, to avoid optional sklearn/TensorFlow/Keras imports.

Outputs:

```text
data/amazon_toys_encoders/interactions.parquet
data/amazon_toys_encoders/item_texts.parquet
data/amazon_toys_encoders/embeddings/*.parquet
```

In [ ]:
build_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.build_amazon_toys_embeddings',
    '--reviews-gz', str(REVIEWS_GZ),
    '--meta-gz', str(META_GZ),
    '--output-dir', str(RAW_OUT),
    '--max-users', '5000',
    '--max-interactions', '3000000',
    '--batch-size', '256',
    '--max-length', '256',
    '--reuse-prepared',
    '--reuse-embeddings',
]

print('+', ' '.join(build_cmd))
subprocess.run(build_cmd, check=True)

## 2. Check Embeddings

Look for non-zero dimension variance and cosine quantiles that are not all `1.0`.

In [ ]:
check_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.check_embeddings',
    '--embeddings-dir', str(RAW_OUT / 'embeddings'),
    '--output', str(RESULTS_DIR / 'embedding_checks.json'),
]

print('+', ' '.join(check_cmd))
subprocess.run(check_cmd, check=True)

## 3. Smoke Test

Tiny run: prepare + fixed dVAE + SID metrics, no seqrec.

In [ ]:
smoke_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.run_encoder_experiment',
    '--interactions', str(RAW_OUT / 'interactions.parquet'),
    '--embeddings-dir', str(RAW_OUT / 'embeddings'),
    '--work-data-dir', 'data/RQ_encoder_comparison/amazon_toys_smoke',
    '--results-dir', 'results/RQ_encoder_comparison_amazon_toys_smoke',
    '--dvae-config', 'configs/RQ_encoder_comparison/amazon_toys_fixed_dvae.yaml',
    '--seqrec-config', 'configs/RQ_encoder_comparison/seqrec_amazon_toys_fixed.yaml',
    '--test-interval-days', '365',
    '--stages', 'prepare,dvae,sid_metrics',
    '--max-users', '1000',
    '--max-core-items', '5000',
    '--num-epochs', '1',
]

print('+', ' '.join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)

## 4. Full Quick Run

Fixed dVAE uses the full local config (`num_epochs=5`). Seqrec is lightweight: `history_budget=128`, `depth=4`, `beam_size=50`.

In [ ]:
full_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.run_encoder_experiment',
    '--interactions', str(RAW_OUT / 'interactions.parquet'),
    '--embeddings-dir', str(RAW_OUT / 'embeddings'),
    '--work-data-dir', 'data/RQ_encoder_comparison/amazon_toys',
    '--results-dir', 'results/RQ_encoder_comparison_amazon_toys',
    '--dvae-config', 'configs/RQ_encoder_comparison/amazon_toys_fixed_dvae.yaml',
    '--seqrec-config', 'configs/RQ_encoder_comparison/seqrec_amazon_toys_fixed.yaml',
    '--test-interval-days', '365',
    '--stages', 'prepare,dvae,sid_metrics,seqrec',
    '--max-users', '5000',
    '--max-core-items', '30000',
]

print('+', ' '.join(full_cmd))
subprocess.run(full_cmd, check=True)

## 5. Collect Results

In [ ]:
collect_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.collect_results',
    '--results-dir', str(RESULTS_DIR),
]

print('+', ' '.join(collect_cmd))
subprocess.run(collect_cmd, check=True)

In [ ]:
import pandas as pd

pd.read_csv(RESULTS_DIR / 'comparison.csv')